<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch15_reliability_missingness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第15回：測定の信頼性と、情報を持つ欠測


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Landis & Koch (1977) kappaの目安／Shrout & Fleiss (1979) ICCの型／音声研究における測定可用性バイアス。

In [ ]:
install.packages(c("irr", "psych", "lme4"), repos = "https://cloud.r-project.org")
library(irr)     # kappa係数
library(psych)   # 級内相関係数 ICC
library(lme4)
library(tidyverse)


■ 分析の前に、測定そのものを疑う：

ここまでの章は、データが正しく測れていることを暗黙の前提にしてきた。しかし音声研究のデータは、人手のアノテーションか自動抽出アルゴリズムを通って生まれる。**どちらも失敗する**。境界の付け方は人によって違い、F0の自動抽出は声質によって失敗する。

この章では2つの問題を扱う。第一に、**複数の人が同じものを測ったとき、どれくらい一致しているか**を数値化する方法。第二に、**測定に失敗したデータが「たまたま」失敗したのではないとき**に何が起きるかである。後者は音声研究に固有と言ってよいほど重要で、しかも見落とされやすい。

■ 1. 評定者間の一致度：一致率をそのまま報告してはいけない：

In [ ]:
set.seed(15)
n_item <- 60

# 「本当の値」を決めて、そこに評定者ごとの癖と誤差を乗せる
truth <- rnorm(n_item, 0, 1)
make_rating <- function(bias, noise) {
  as.integer(cut(truth + bias + rnorm(n_item, 0, noise),
                 breaks = c(-Inf, -1, 0, 1, Inf), labels = 1:4))
}
ratings <- data.frame(
  r1 = make_rating(0,   0.4),
  r2 = make_rating(0,   0.4),
  r3 = make_rating(0.6, 0.4)   # r3 は全体的に甘めに付ける評定者
)

cat("単純一致率 r1-r2      :", round(mean(ratings$r1 == ratings$r2), 3), "\n")
cat("Cohen's kappa r1-r2   :", round(kappa2(ratings[, c("r1","r2")])$value, 3), "\n")
cat("Cohen's kappa r1-r3   :", round(kappa2(ratings[, c("r1","r3")])$value, 3), "\n")
cat("重み付きkappa r1-r3   :",
    round(kappa2(ratings[, c("r1","r3")], weight = "squared")$value, 3), "\n")

cat("\n級内相関係数 ICC:\n")
# ★ICC2（単一評定者、two-way random、絶対一致）とICC2k（3人平均、
#   同じtwo-way randomモデルのk人平均版）を並べる。psych::ICC()の結果は
#   [1]=ICC1, [2]=ICC2, [3]=ICC3, [4]=ICC1k, [5]=ICC2k, [6]=ICC3k の順で
#   返るので、「単一評定者→その平均」を比べるにはモデルの型を揃えて
#   [2]と[5]を選ぶ。[2]と[4]（ICC1k）を並べると、two-way randomモデルの
#   単一評定者値と、別モデルであるone-way randomの平均値を比べることに
#   なってしまう——見かけ上は近い値が出ることがあるが、モデルの前提
#   （評定者を無作為抽出とみなすかどうかの一方向/二方向の違い）が違うため、
#   同じ「平均化の効果」を測ったことにならない。
print(ICC(ratings)$results[c(2, 5), c("type", "ICC", "lower bound", "upper bound")])

# --- 実行結果 ---
# 単純一致率 r1-r2      : 0.55
# Cohen's kappa r1-r2   : 0.391
# Cohen's kappa r1-r3   : 0.211
# 重み付きkappa r1-r3   : 0.622
#
# 級内相関係数 ICC:
#                        type       ICC lower bound upper bound
# Single_random_raters   ICC2 0.6592413   0.4566045   0.7912765
# Average_random_raters ICC2k 0.8530251   0.7159770   0.9191794
#
# 【出力の読み方】
# ・**単純一致率55%を「まあまあ一致している」と読んではいけない。**
#   4段階評定なら、2人がでたらめに付けても25%は偶然一致する。
#   一致率は偶然の一致を含んだ値なので、そのままでは解釈できない。
# ・Cohen's kappaは**偶然の一致を差し引いた**指標である。r1-r2で0.391。
#   慣習的な目安は 0.41-0.60 で「中程度」、0.61-0.80 で「かなり良い」、
#   0.81以上で「ほぼ完全」とされる（Landis & Kochの基準）。
#   0.391は「まずまず」程度で、一致率55%という見かけよりずっと厳しい評価になる。
# ・r1-r3のkappaは0.211とさらに低い。r3は「全体的に甘めに付ける」評定者として
#   作ったので、体系的にずれている。**kappaは体系的なずれに厳しい。**
# ・★ところが**重み付きkappaにすると0.622まで跳ね上がる**。
#   通常のkappaは「1と2」の食い違いも「1と4」の食い違いも同じ1つの不一致として
#   数えるが、順序のある評定ではこれは不当である。隣の段への食い違いは
#   軽い誤りであるべきだからである。重み付きkappaはずれの大きさに応じて
#   罰則を変える。**順序尺度の評定では、必ず重み付きkappaを使う。**
#   通常のkappaを使うと、一致度を大幅に過小評価する。
# ・ICCは評定を連続量とみなしたときの一致度である。2つの値が出ている点に注意する。
#   ICC2（0.659）は**評定者1人あたり**の信頼性、ICC2k（0.853）は
#   **3人の平均値**の信頼性である。★どちらも「two-way random、絶対一致」
#   という同じモデルの単一評定者版・k人平均版なので、この2つを比べて初めて
#   「平均化することで信頼性がどれだけ上がるか」を正しく測ったことになる
#   （型の違うICC1kと比べると、平均化の効果とモデルの違いが混ざってしまう）。
#   ★どちらを報告するかは、**実際の分析で何を使うか**で決まる。
#   3人の平均値を分析に使うならICC2kを、1人ずつの評定を使うならICC2を報告する。
#   平均値を使うのに単一評定者のICCを報告すると、不当に低く見せることになる。
# ・**評定者を増やすと平均値の信頼性は上がる**（0.659 → 0.853）。
#   これが「評定者は3人以上」と言われる根拠である。信頼区間も併記されており、
#   ICC2は0.457から0.791と幅が広い。60項目でもこの程度の精度しか出ない。


■ 2. ★情報を持つ欠測：測定の失敗が条件と相関するとき：

F0の自動抽出は、声門振動が不規則な区間で失敗する。つまり**きしみ声（creaky voice）では失敗しやすい**。そして、きしみ声はF0が低いときに起こりやすい。したがって**「測れなかったデータ」は低いF0に偏っている**。この状況で、測れたデータだけを平均すると何が起きるかを確かめる。

In [ ]:
set.seed(16)
n_speaker <- 20
n_item    <- 40

f0_data <- expand_grid(speaker = sprintf("S%02d", 1:n_speaker), item = 1:n_item) %>%
  mutate(
    cond = rep(c("modal", "creaky"), length.out = n()),
    # 真のF0: creaky条件では25Hz低い
    f0_true = 180 + if_else(cond == "creaky", -25, 0) +
              rnorm(n_speaker, 0, 15)[as.integer(factor(speaker))] +
              rnorm(n(), 0, 12),
    # ★抽出の失敗確率は F0 が低いほど高い（条件ではなくF0の値で決まる）
    p_fail = plogis(-1.0 - 0.10 * (f0_true - 180)),
    failed = rbinom(n(), 1, p_fail),
    f0_obs = if_else(failed == 1, NA_real_, f0_true)
  )

cat("抽出失敗率: modal", round(mean(f0_data$failed[f0_data$cond == "modal"]), 3),
    " / creaky", round(mean(f0_data$failed[f0_data$cond == "creaky"]), 3), "\n")
cat("全体の欠測率:", round(mean(f0_data$failed), 3), "\n\n")

true_m <- mean(f0_data$f0_true[f0_data$cond == "modal"])
true_c <- mean(f0_data$f0_true[f0_data$cond == "creaky"])
obs <- f0_data %>% filter(!is.na(f0_obs))
obs_m <- mean(obs$f0_obs[obs$cond == "modal"])
obs_c <- mean(obs$f0_obs[obs$cond == "creaky"])

cat(sprintf("真の値  : modal %.2f  creaky %.2f  差 %.2f\n", true_m, true_c, true_m - true_c))
cat(sprintf("観測値  : modal %.2f  creaky %.2f  差 %.2f\n", obs_m, obs_c, obs_m - obs_c))
cat(sprintf("差の過小評価: %.1f%%\n", 100 * (1 - (obs_m - obs_c) / (true_m - true_c))))

cat("\n真値で当てはめたモデルの係数  :",
    round(fixef(lmer(f0_true ~ cond + (1 | speaker), data = f0_data))["condmodal"], 2), "\n")
cat("観測値で当てはめたモデルの係数:",
    round(fixef(lmer(f0_obs  ~ cond + (1 | speaker), data = f0_data))["condmodal"], 2), "\n")

# --- 実行結果 ---
# 抽出失敗率: modal 0.28  / creaky 0.652
# 全体の欠測率: 0.466
#
# 真の値  : modal 183.82  creaky 159.93  差 23.89
# 観測値  : modal 190.55  creaky 174.23  差 16.32
# 差の過小評価: 31.7%
#
# 真値で当てはめたモデルの係数  : 23.89
# 観測値で当てはめたモデルの係数: 20.15
#
# 【出力の読み方】
# ・**両群とも観測された平均が真の平均より高い。** modalは183.82→190.55、
#   creakyは159.93→174.23。低いF0のトークンが優先的に失われるので、
#   残ったものだけを平均すれば当然そうなる。
# ・重要なのは**上がり方が群によって違う**ことである。creaky条件のほうが
#   失敗率が高い（0.652 対 0.280）ため、平均の押し上げも大きい
#   （+14.30 対 +6.73）。結果として**条件間の差が23.89から16.32へ、
#   31.7%も縮んでいる**。
# ・これは分析手法の問題ではない。混合効果モデルを使っても係数は
#   23.89から20.15へ縮む。**どんな高度なモデルも、失われたデータは復元できない。**
# ・★この状況を「情報を持つ欠測」（informative missingness）と呼ぶ。
#   欠測が完全にランダム（MCAR）なら、標本が減るだけで推定値は歪まない。
#   しかしここでは**欠測するかどうかが、測ろうとしている値そのものに依存する**。
#   これは統計学でMNAR（missing not at random）と呼ばれる最も厄介な型である。
# ・**音声研究ではこの型の欠測が例外ではなく標準である。** 代表例を挙げる。
#   - F0抽出はきしみ声・ささやき声で失敗する（F0が低い/不規則なときに失敗）
#   - フォルマント推定は高いF0の話者（女性・子ども）で不安定になる
#   - 強制アライメントは発話速度が速い区間や不明瞭な区間でずれる
#   - VOTの測定は、閉鎖が弱い場合に境界を決められない
#   いずれも**測りにくい条件そのものが研究対象**であることが多い。
# ・★対処は3段階で考える。
#   (1) **必ず欠測率を報告する。** 条件ごと・話者ごとに集計して示す。
#       上の例なら「creaky条件で65%が抽出に失敗した」と書く。この一文があれば、
#       読者は結果を割り引いて読める。**黙って除外するのが最悪である。**
#   (2) **欠測が条件と関係しているかを検定する。**
#       glm(failed ~ cond, family = binomial) を当てればよい。
#       関係していれば、単純な除外は使えないと分かる。
#   (3) **失敗そのものを結果として扱うことを検討する。**
#       「creaky条件では65%でF0が抽出できなかった」という事実は、
#       きしみ声の存在を示す立派な証拠である。測れた分だけを平均するより、
#       抽出可否を二値の応答としてGLMM（第6回）にかけるほうが、
#       研究の問いに答えていることがある。
# ・★最後の点が最も重要である。**測定の失敗をノイズとして捨てるのではなく、
#   データとして読む。** 測れなかったことにも意味がある。


■ 演習：

1. 欠測が条件と関係しているかを `glm(failed ~ cond, family = binomial)` で検定せよ。
2. 抽出失敗を応答変数とするGLMMを当てはめ、「creaky条件では抽出に失敗しやすい」ことを結果として報告する形にせよ。
3. 評定者 r3 を除いて ICC を計算し直し、値がどう変わるかを確かめよ。

■ 演習の解答：

In [ ]:
# 1. 欠測が条件に依存しているかを検定する。
glm_miss <- glm(failed ~ cond, family = binomial, data = f0_data)
print(round(summary(glm_miss)$coefficients, 4))
cat("オッズ比:", round(exp(coef(glm_miss)["condmodal"]), 3),
    "→ modal条件はcreaky条件に比べて失敗のオッズがこの倍率\n")

# 2. 「測れなかったこと」自体を結果として分析する。
glmm_miss <- glmer(failed ~ cond + (1 | speaker), family = binomial, data = f0_data)
cat("\n抽出失敗のGLMM:\n")
print(round(summary(glmm_miss)$coefficients, 4))
cat("creaky条件の失敗確率:", round(plogis(fixef(glmm_miss)["(Intercept)"]), 3), "\n")
cat("modal条件の失敗確率  :", round(plogis(sum(fixef(glmm_miss))), 3), "\n")

# 3. 甘めの評定者r3を除いてICCを計算する。
cat("\n3人全員のICC(単一評定者):",
    round(ICC(ratings)$results[2, "ICC"], 3), "\n")
cat("r1,r2のみのICC(単一評定者):",
    round(ICC(ratings[, c("r1","r2")])$results[2, "ICC"], 3), "\n")

# --- 実行結果 ---
#             Estimate Std. Error  z value Pr(>|z|)
# (Intercept)   0.6300     0.1050   6.0003        0
# condmodal    -1.5745     0.1531 -10.2871        0
# オッズ比: 0.207 → modal条件はcreaky条件に比べて失敗のオッズがこの倍率
#
# 抽出失敗のGLMM:
#             Estimate Std. Error  z value Pr(>|z|)
# (Intercept)   0.8533     0.3053   2.7952   0.0052
# condmodal    -2.0882     0.1918 -10.8863   0.0000
# creaky条件の失敗確率: 0.701
# modal条件の失敗確率  : 0.225
#
# 3人全員のICC(単一評定者): 0.659
# r1,r2のみのICC(単一評定者): 0.769
#
# 【出力の読み方】
# ・1問目: condmodal の係数は −1.5745、p はほぼ0。オッズ比0.207は
#   「modal条件では失敗のオッズがcreaky条件の約5分の1」という意味である。
#   **欠測は明確に条件と結びついている。** したがって
#   「測れた分だけを分析する」という方針はこの時点で使えないと分かる。
#   ★この検定は1行で書けるのに、実際にはほとんど行われない。
#   欠測を見つけたら、まずこれを走らせる習慣をつけたい。
# ・2問目: 失敗そのものを応答としたGLMMでは、creaky条件の失敗確率が0.701、
#   modal条件が0.225と推定された。**この2つの数値は、それ自体が
#   報告に値する結果である。** 「creaky条件では約70%でF0が抽出できなかった」
#   という記述は、きしみ声が実際に生じていたことの客観的な証拠になる。
# ・話者のランダム切片を入れているので、「特定の話者だけが失敗を稼いでいる」
#   可能性も調整済みである。切片のp = 0.0052 は
#   「creaky条件の失敗確率が50%と異なる」ことを示す。
# ・★研究の書き方が変わる点に注目したい。単純な報告なら
#   「F0の差は16.3Hzだった（真値は23.9Hz）」と、31.7%過小評価した値を
#   そのまま書いてしまう。しかし欠測を分析すれば、
#   「creaky条件では70%で抽出に失敗しており、測れたトークンは
#   相対的にF0が高いものに偏る。したがって16.3Hzという差は
#   真の差の下限とみなすべきである」と書ける。
#   **同じデータから、はるかに誠実で情報量の多い記述が得られる。**
# ・3問目: 甘めの評定者r3を外すと、単一評定者のICCは0.659から0.769へ上がる。
#   r3が体系的にずれていたぶん、一致度が下がっていたことが確認できる。
# ・★ただし**「一致度が上がるから外す」という判断はしてはいけない。**
#   それはデータを結果に合わせて選ぶことであり、第12回で見た
#   「有意になった比較だけを報告する」のと同じ問題である。
#   評定者を外してよいのは、手順の誤解や作業ミスなど、
#   **一致度とは独立した理由が確認できたとき**に限る。
#   外した場合は、外した理由と、外す前後の両方の値を報告する。
# ・（`r1,r2のみ` のICCを計算する際に "boundary (singular) fit" という警告が
#   出ることがある。評定者2人だと分散成分の推定が境界に貼り付きやすいためで、
#   ICCの値そのものは使える。第5回で扱ったsingular fitと同じ現象である。）
